# Network Expansion
## Model 3 (Test) - Stochastic Intertemporal Model

Scenario-based, multi-year formulation: investments (substations, lines, reinforcements) are shared across scenarios and decided once; operations are scenario-specific. Costs are discounted and budgets are enforced per year. Uncertainty enters via demand scenarios with probabilities.

### 1 - Imports

In [8]:
import numpy as np
import pandas as pd

from src.classes import DistributionNetwork, Substation
from src.solver import solve_network_stochastic

### 2 - Define the Distribution Network (shared)

In [9]:
# Nodes and loads
NODES = [f"N{i}" for i in range(1, 14)]
LOADS = [f"D{i}" for i in range(1, 11)]

# Initial substation
S1 = Substation("S1", "N4", 40, ['N3', 'N5', 'N9'], r_cost=200, edge_cost=50)
SUBSTATIONS = [S1]

line_cost = 50

base_load_capacity = {
    'D1': 6, 'D2': 3, 'D3': 2, 'D4': 5, 'D5': 3,
    'D6': 2, 'D7': 3, 'D8': 5, 'D9': 4, 'D10': 6
}

loads_locations = {
    'D1': 'N1', 'D2': 'N2', 'D3': 'N3', 'D4': 'N6', 'D5': 'N7',
    'D6': 'N8', 'D7': 'N9', 'D8': 'N11', 'D9': 'N12', 'D10': 'N13'
}

nodes_connected = {
    'N1': ['N2'],
    'N2': ['N1', 'N3'],
    'N3': ['N2', 'N4'],
    'N4': ['N3', 'N5', 'N9'],
    'N5': ['N4', 'N6'],
    'N6': ['N5','N7', 'N8'],
    'N7': ['N6'],
    'N8': ['N6'],
    'N9': ['N4', 'N10'],
    'N10': ['N9', 'N11', 'N13'],
    'N11': ['N10', 'N12'],
    'N12': ['N11'],
    'N13': ['N10']
}

DistributionNetwork = DistributionNetwork(
    NODES.copy(),
    LOADS.copy(),
    SUBSTATIONS.copy(),
    base_load_capacity.copy(),
    nodes_connected.copy(),
    loads_locations.copy(),
    line_cost
)

# Candidate substations
capacity = 15
s_cost = 100       # activation cost
l_cost = line_cost # feeder line cost
r_cost = 200       # capacity reinforcement cost

S2 = Substation("S2", "N14", capacity, ['N2'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S3 = Substation("S3", "N15", capacity, ['N6'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S4 = Substation("S4", "N16", capacity, ['N11', 'N13'], r_cost, edge_cost=l_cost, fix_cost=s_cost)

DistributionNetwork.add_candidate_substations([S2, S3, S4])

### 3 - Scenarios and horizon

In [10]:
years = list(range(1, 11))      # Years 1..10
R = 10                             # Capacity reinforcement size
dr = 0.05                          # Discount rate
# Dynamic budgets per year
B = {t: 550 + 25*(t-1) for t in years}

# Scenario growth rates (annual) and probabilities
scenarios_def = {
    'conservative': {'prob': 0.25, 'growth': 0.01},
    'base':         {'prob': 0.50, 'growth': 0.04},
    'high':         {'prob': 0.25, 'growth': 0.07},
}

# Build per-scenario, per-year demands
scenarios = {}
for name, data in scenarios_def.items():
    g = data['growth']
    scenarios[name] = {'prob': data['prob'], 'demands': {}}
    for t in years:
        factor = (1 + g) ** (t - 1)
        scenarios[name]['demands'][t] = {ld: val * factor for ld, val in base_load_capacity.items()}

# Quick demand check
total_demand = {name: {t: round(sum(scenarios[name]['demands'][t].values()), 2) for t in years} for name in scenarios}
total_demand

{'conservative': {1: 39.0,
  2: 39.39,
  3: 39.78,
  4: 40.18,
  5: 40.58,
  6: 40.99,
  7: 41.4,
  8: 41.81,
  9: 42.23,
  10: 42.65},
 'base': {1: 39.0,
  2: 40.56,
  3: 42.18,
  4: 43.87,
  5: 45.62,
  6: 47.45,
  7: 49.35,
  8: 51.32,
  9: 53.37,
  10: 55.51},
 'high': {1: 39.0,
  2: 41.73,
  3: 44.65,
  4: 47.78,
  5: 51.12,
  6: 54.7,
  7: 58.53,
  8: 62.63,
  9: 67.01,
  10: 71.7}}

### 4 - Solve stochastic model

In [11]:
solution = solve_network_stochastic(
    Network=DistributionNetwork,
    R=R,
    B=B,
    dr=dr,
    years=years,
    scenarios=scenarios,
    OutputFlag=0
)

### 5 - Investment summary (scenario-invariant)

In [12]:
S_idx = list(range(1, len(DistributionNetwork.SUBSTATIONS)+1))
base_edge_cost = DistributionNetwork.edge_cost

lines_added = {}
for t in years:
    added = set()
    for (i, j, s, tau), val in solution['b_on'].items():
        if tau == t and val > 0.5:
            e = (i, j)
            if base_edge_cost.get(e, 0) > 0:
                added.add(e)
    lines_added[t] = sorted(list(added))

investment_summary = pd.DataFrame({
    'Budget': [B[t] if not isinstance(B, dict) else B[t] for t in years],
    'Substations Active': [[s for s in S_idx if solution['w'][(s, t)] > 0.5] for t in years],
    'Lines Added': [lines_added[t] for t in years],
    'Cost substation (nom)': [round(solution['cost_components_nominal'][t]['substation'], 2) for t in years],
    'Cost lines (nom)': [round(solution['cost_components_nominal'][t]['lines'], 2) for t in years],
    'Cost reinforcement (nom)': [round(solution['cost_components_nominal'][t]['reinforcement'], 2) for t in years],
    'Cost (nominal)': [round(solution['cost_per_year_nominal'][t], 2) for t in years],
    'Cost (discounted)': [round(solution['cost_per_year'][t], 2) for t in years]
}, index=years)

investment_summary

,Budget,Substations Active,Lines Added,Cost substation (nom),Cost lines (nom),Cost reinforcement (nom),Cost (nominal),Cost (discounted)
1,550,[1],[],0.0,0.0,0.0,0.0,0.00
2,575,"[1, 4]","[(11, 16)]",100.0,50.0,0.0,150.0,142.86
3,600,"[1, 4]",[],0.0,0.0,0.0,0.0,0.00
4,625,"[1, 4]",[],0.0,0.0,0.0,0.0,0.00
5,650,"[1, 4]",[],0.0,0.0,0.0,0.0,0.00
6,675,"[1, 4]",[],0.0,0.0,200.0,200.0,156.71
7,700,"[1, 4]",[],0.0,0.0,0.0,0.0,0.00
8,725,"[1, 4]",[],0.0,0.0,0.0,0.0,0.00
9,750,"[1, 4]",[],0.0,0.0,200.0,200.0,135.37
10,775,"[1, 4]",[],0.0,0.0,0.0,0.0,0.00


### 6 - Scenario operations

In [13]:
records = []
for omega, data in scenarios.items():
    for t in years:
        demand = sum(data['demands'][t].values())
        supply = sum(solution['r'][(s, t, omega)] for s in S_idx)
        active = [s for s in S_idx if solution['w'][(s, t)] > 0.5]
        records.append({
            'Scenario': omega,
            'Prob': data['prob'],
            'Year': t,
            'Demand': round(demand, 2),
            'Supply': round(supply, 2),
            'Substations Active': active
        })

ops_summary = pd.DataFrame(records)
ops_summary

,Scenario,Prob,Year,Demand,Supply,Substations Active
0,conservative,0.25,1,39.00,39.00,[1]
1,conservative,0.25,2,39.39,39.39,"[1, 4]"
2,conservative,0.25,3,39.78,39.78,"[1, 4]"
3,conservative,0.25,4,40.18,40.18,"[1, 4]"
4,conservative,0.25,5,40.58,40.58,"[1, 4]"
5,conservative,0.25,6,40.99,40.99,"[1, 4]"
6,conservative,0.25,7,41.40,41.40,"[1, 4]"
7,conservative,0.25,8,41.81,41.81,"[1, 4]"
8,conservative,0.25,9,42.23,42.23,"[1, 4]"
9,conservative,0.25,10,42.65,42.65,"[1, 4]"


### 7 - Final-year assignments per scenario

In [14]:
last_year = years[-1]
S_idx = list(range(1, len(DistributionNetwork.SUBSTATIONS)+1))
N_idx = list(range(1, len(DistributionNetwork.NODES)+1))
for omega in scenarios:
    print(f"Assignments in Year {last_year} for scenario {omega}:")
    for n in N_idx:
        for s in S_idx:
            if solution['y'][(n, s, last_year, omega)] > 0.5:
                print(f"Node {DistributionNetwork.NODES[n-1]} assigned to {DistributionNetwork.SUBSTATIONS[s-1].id}")


Assignments in Year 10 for scenario conservative:
Node N1 assigned to S1
Node N2 assigned to S1
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S4
Node N12 assigned to S4
Node N13 assigned to S1
Node N16 assigned to S4
Assignments in Year 10 for scenario base:
Node N1 assigned to S1
Node N2 assigned to S1
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S4
Node N10 assigned to S4
Node N11 assigned to S4
Node N12 assigned to S4
Node N13 assigned to S4
Node N16 assigned to S4
Assignments in Year 10 for scenario high:
Node N1 assigned to S1
Node N2 assigned to S1
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S4
Node 